# 🚨 Quantum Machine Learning for Fraud Detection
### Quantum for Finance — Quantum for Humanity

This notebook applies **Quantum Support Vector Machines (QSVM)** and **Quantum Kernel Estimation** to financial transaction fraud detection.

**Learning source:** [IBM Quantum Learning — Machine Learning](https://learning.quantum.ibm.com)

---
## Background

Quantum kernel methods use quantum feature maps $\phi(x)$ to map classical data into a high-dimensional Hilbert space, potentially capturing patterns classical kernels cannot. For fraud detection:

$$K(x_i, x_j) = |\langle \phi(x_i) | \phi(x_j) \rangle|^2$$

The ZZFeatureMap from Qiskit creates entangled feature encodings that may outperform classical RBF kernels on certain financial datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute

print('✅ Imports successful')

## Step 1: Generate Synthetic Transaction Dataset

In [ ]:
# Simulate financial transaction features
# Features: [transaction_amount, time_of_day, merchant_category, distance_from_home,
#            velocity_24h, account_age_months]

np.random.seed(42)
X, y = make_classification(
    n_samples=200,
    n_features=4,     # reduced for quantum (fewer qubits needed)
    n_informative=3,
    n_redundant=1,
    n_clusters_per_class=1,
    weights=[0.9, 0.1],   # 10% fraud (realistic class imbalance)
    flip_y=0.01,
    random_state=42
)

feature_names = ['Transaction Amount', 'Time of Day', 'Merchant Risk Score', 'Velocity 24h']
print(f'Dataset: {X.shape[0]} transactions, {X.shape[1]} features')
print(f'Class distribution — Legitimate: {(y==0).sum()}, Fraudulent: {(y==1).sum()}')

# Standardise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Step 2: Classical SVM Baseline

In [ ]:
# Classical RBF SVM baseline
classical_svm = SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=42)
classical_svm.fit(X_train, y_train)
y_pred_classical = classical_svm.predict(X_test)

print('=== Classical RBF SVM ===')
print(classification_report(y_test, y_pred_classical, target_names=['Legitimate', 'Fraud']))

## Step 3: Quantum Kernel SVM (QSVC)

In [ ]:
# Build ZZFeatureMap quantum kernel (4 features → 4 qubits)
num_features = X_train.shape[1]
feature_map = ZZFeatureMap(feature_dimension=num_features, reps=2, entanglement='linear')
print(f'Quantum feature map: {num_features} qubits, depth {feature_map.depth()}')
feature_map.draw('mpl', style='clifford')

In [ ]:
# Build quantum kernel using fidelity estimation
sampler = Sampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

# Train QSVC
# Note: use a small subset for simulation speed
train_size = min(60, len(X_train))
test_size  = min(30, len(X_test))

qsvc = QSVC(quantum_kernel=quantum_kernel, C=1.0, class_weight='balanced')
qsvc.fit(X_train[:train_size], y_train[:train_size])
y_pred_quantum = qsvc.predict(X_test[:test_size])

print('=== Quantum Kernel SVM (QSVC) ===')
print(classification_report(y_test[:test_size], y_pred_quantum, target_names=['Legitimate', 'Fraud']))

## Step 4: Compare and Visualise Results

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Classical
cm_classical = confusion_matrix(y_test, y_pred_classical)
ConfusionMatrixDisplay(cm_classical, display_labels=['Legit', 'Fraud']).plot(
    ax=axes[0], colorbar=False, cmap='Purples')
axes[0].set_title('Classical RBF SVM', fontsize=13, fontweight='bold')

# Quantum
cm_quantum = confusion_matrix(y_test[:test_size], y_pred_quantum)
ConfusionMatrixDisplay(cm_quantum, display_labels=['Legit', 'Fraud']).plot(
    ax=axes[1], colorbar=False, cmap='RdPu')
axes[1].set_title('Quantum Kernel SVM (QSVC)', fontsize=13, fontweight='bold')

plt.suptitle('Fraud Detection — Classical vs Quantum SVM', fontsize=14)
plt.tight_layout()
plt.show()

## 🌍 Humanitarian Application

Quantum fraud detection can protect the most vulnerable:
- **Mobile money fraud** (M-Pesa, UPI) targeting low-income users
- **Microfinance loan fraud** in developing markets
- **Digital payment security** for first-time banking customers

Better fraud detection → lower losses → lower costs → financial services reach more people.

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*